## Creating Graph on FarFetch Outfits Dataset


In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

tqdm.pandas()

import itertools
import os

In [ ]:
prod_df = pd.read_parquet('data/products.parquet')

In [ ]:
prod_df = prod_df.reset_index()
prod_df = prod_df.rename(columns={"index":'product_node_id'})
prod_df

In [ ]:
selected_columns = ['product_node_id', 'product_id', 'product_family', 'product_category', 'product_sub_category', 'product_gender',
                            'product_main_colour', 'product_brand', 'product_materials', 'product_short_description', 'product_image_path',
                            'product_highlights']
prod_df[selected_columns].to_parquet('data/graph/products.parquet', index=False)

In [ ]:
product_family_map = prod_df.set_index('product_id')['product_family'].to_dict()
product_category_map = prod_df.set_index('product_id')['product_category'].to_dict()
product_subcategory_map = prod_df.set_index('product_id')['product_sub_category'].to_dict()

## Product Family Nodes

In [ ]:
prod_family_df = prod_df[['product_family']].drop_duplicates().reset_index(drop=True).reset_index()
prod_family_df.columns = ['family_node_id', 'product_family']
prod_family_df

In [ ]:
#product_family_node_dict = prod_family_df.set_index('family_node_id')['family_name'].to_dict()
#product_family_node_dict

In [ ]:
prod_family_df.to_parquet('data/graph/prod_family.parquet', index=False)

## Product Category Nodes

In [ ]:
prod_cat_df = prod_df[['product_category']].drop_duplicates().reset_index(drop=True).reset_index()
prod_cat_df.columns = ['cat_node_id', 'product_category']
prod_cat_df

In [ ]:
#product_category_map = prod_df.set_index('product_id')['product_category'].to_dict()


In [ ]:
prod_cat_df.to_parquet('data/graph/prod_category.parquet', index=False)

## Family - Category Edges

In [ ]:
fam_cat_edge_df = prod_df[['product_family', 'product_category']].drop_duplicates().reset_index(drop=True)
fam_cat_edge_df

In [ ]:
fam_cat_edge_df = fam_cat_edge_df.merge(prod_family_df, on='product_family', how='left').merge(prod_cat_df, on='product_category', how='left')
fam_cat_edge_df

In [ ]:
fam_cat_edge_df.to_parquet('data/graph/fam_cat_edge.parquet', index=False)

## Category - Product Edge

In [ ]:
cat_prod_edge_df = prod_df[['product_category', 'product_id', 'product_node_id']].merge(prod_cat_df, on='product_category', how='left')
cat_prod_edge_df

In [ ]:
cat_prod_edge_df.to_parquet('data/graph/cat_prod_edge.parquet', index=False)

# Outfits

In [ ]:
product_family_map = prod_df.set_index('product_id')['product_family'].to_dict()
product_category_map = prod_df.set_index('product_id')['product_category'].to_dict()
product_subcategory_map = prod_df.set_index('product_id')['product_sub_category'].to_dict()

In [7]:
outfits_df = pd.read_parquet('data/manual_outfits.parquet')
outfits_df.shape

(300000, 2)

In [ ]:
i=10
pd.DataFrame({
    'Family_composition': pd.Series(outfits_df['products'][i]).map(product_family_map),
    'category_composition': pd.Series(outfits_df['products'][i]).map(product_category_map),
    'subcategory_composition': pd.Series(outfits_df['products'][i]).map(product_subcategory_map)
})

In [10]:
# split outfits into train, val and test set

train_df = outfits_df.sample(frac=0.9)

print(f"Training shape:{train_df.shape}")

val_test_df = outfits_df.drop(train_df.index)

val_df = val_test_df.sample(frac=0.5)
test_df = val_test_df.drop(val_df.index)

print(f"Validation shape:{val_df.shape}")
print(f"Testing shape:{test_df.shape}")

train_df.to_parquet("data/training_outfits.parquet")
val_df.to_parquet("data/validation_outfits.parquet")
test_df.to_parquet("data/testing_outfits.parquet")


Training shape:(270000, 2)
Validation shape:(15000, 2)
Testing shape:(15000, 2)


In [ ]:
train_outfits_df = pd.read_parquet("data/training_outfits.parquet")
train_outfits_df

## Outfit - Family edge

In [ ]:
outfits_df = train_outfits_df

In [ ]:
outfits_edges_df = outfits_df['products'].apply(lambda x: list(itertools.combinations(x,2)))
outfits_edges_df = pd.DataFrame(outfits_edges_df.explode('products'))
outfits_edges_df[['p1', 'p2']] = pd.DataFrame(outfits_edges_df['products'].to_list(), index=outfits_edges_df.index)
outfits_edges_df

In [ ]:
outfits_edges_df['category_p1'] = outfits_edges_df['p1'].map(product_category_map)
outfits_edges_df['category_p2'] = outfits_edges_df['p2'].map(product_category_map)
outfits_edges_df['family_p1'] = outfits_edges_df['p1'].map(product_family_map)
outfits_edges_df['family_p2'] = outfits_edges_df['p2'].map(product_family_map)
outfits_edges_df

In [ ]:
pcat_nodeid_map = prod_cat_df.set_index('product_category')['cat_node_id'].to_dict()
pfam_nodeid_map = prod_family_df.set_index('product_family')['family_node_id'].to_dict()
p_nodeid_map = prod_df.set_index('product_id')['product_node_id'].to_dict()

In [ ]:
outfits_edges_df['p1_node_id'] = outfits_edges_df['p1'].map(p_nodeid_map)
outfits_edges_df['p2_node_id'] = outfits_edges_df['p2'].map(p_nodeid_map)

outfits_edges_df['category_p1_node_id'] = outfits_edges_df['category_p1'].map(pcat_nodeid_map)
outfits_edges_df['category_p2_node_id'] = outfits_edges_df['category_p2'].map(pcat_nodeid_map)

outfits_edges_df['family_p1_node_id'] = outfits_edges_df['family_p1'].map(pfam_nodeid_map)
outfits_edges_df['family_p2_node_id'] = outfits_edges_df['family_p2'].map(pfam_nodeid_map)
outfits_edges_df

In [ ]:
outfits_edges_df[['p1_node_id', 'p2_node_id',
                  'category_p1_node_id', 'category_p2_node_id',
                  'family_p1_node_id', 'family_p2_node_id']].to_parquet("data/graph/outfits_edges.parquet", index=False)

# Graph

In [1]:
import pandas as pd

import torch
from torch_geometric.data import Data, HeteroData
import torch_geometric.transforms as T

In [2]:
#fam_cat_edge_df = pd.read_parquet('data/graph/fam_cat_edge.parquet')
#cat_prod_edge_df = pd.read_parquet('data/graph/cat_prod_edge.parquet')
outfits_edges_df = pd.read_parquet("data/graph/outfits_edges.parquet")

In [3]:
prod_family_df = pd.read_parquet('data/graph/prod_family.parquet')
prod_cat_df = pd.read_parquet('data/graph/prod_category.parquet')
prod_df = pd.read_parquet('data/graph/products.parquet', columns=['product_id', 'product_node_id', 'product_family', 'product_category'])

In [4]:
prod_df

,product_id,product_node_id,product_family,product_category
0,17073270,0,Clothing,Knitwear
1,17674562,1,Clothing,Knitwear
2,17678603,2,Clothing,Knitwear
3,17179699,3,Clothing,Knitwear
4,15907453,4,Clothing,Sweaters & Knitwear
...,...,...,...,...
398665,18020113,398665,Clothing,Denim
398666,18151251,398666,Clothing,Denim
398667,18161372,398667,Clothing,Denim
398668,18164968,398668,Clothing,Denim


In [5]:
prod_df = prod_df.merge(prod_cat_df, on='product_category', how='left')
prod_df = prod_df.merge(prod_family_df, on='product_family', how='left')
prod_df

,product_id,product_node_id,product_family,product_category,cat_node_id,family_node_id
0,17073270,0,Clothing,Knitwear,0,0
1,17674562,1,Clothing,Knitwear,0,0
2,17678603,2,Clothing,Knitwear,0,0
3,17179699,3,Clothing,Knitwear,0,0
4,15907453,4,Clothing,Sweaters & Knitwear,1,0
...,...,...,...,...,...,...
398665,18020113,398665,Clothing,Denim,6,0
398666,18151251,398666,Clothing,Denim,6,0
398667,18161372,398667,Clothing,Denim,6,0
398668,18164968,398668,Clothing,Denim,6,0


In [6]:
prod_cat_df

,cat_node_id,product_category
0,0,Knitwear
1,1,Sweaters & Knitwear
2,2,Fine Watches
3,3,Performance Shorts
4,4,Trousers
...,...,...
124,124,Pumps
125,125,Hair Accessories
126,126,Umbrellas
127,127,Music


In [7]:
graph = HeteroData()

In [8]:
#graph['family'].x = torch.randn(prod_family_df.shape[0], 16)
#graph['family'].num_nodes = prod_family_df.shape[0]
#graph['family'].node_id = torch.tensor(prod_family_df['family_node_id'].values)

#graph['category'].x = torch.randn(prod_cat_df.shape[0], 16)
graph['category'].num_nodes = prod_cat_df.shape[0]
graph['category'].node_id = torch.tensor(prod_cat_df['cat_node_id'].values)

#graph['product'].x = torch.randn(prod_df.shape[0], 16)
graph['product'].num_nodes = prod_df.shape[0]
graph['product'].node_id = torch.tensor(prod_df['product_node_id'].values)
graph['product'].category_id = torch.tensor(prod_df['cat_node_id'].values)
graph['product'].family_id = torch.tensor(prod_df['family_node_id'].values)


In [9]:
#graph["family", "fam_cat_link", "category"].edge_index = torch.as_tensor(fam_cat_edge_df[['family_node_id', 'cat_node_id']].values.tolist()).t().contiguous()
#graph["category", "fam-cat-link", "family"].edge_index = torch.as_tensor(fam_cat_edge_df[['cat_node_id', 'family_node_id']].values.tolist()).t().contiguous()

#graph["category", "cat_prod_link", "product"].edge_index = torch.as_tensor(cat_prod_edge_df[['cat_node_id', 'product_node_id']].values.tolist()).t().contiguous()
graph["product", "cat_prod_link", "category"].edge_index = torch.as_tensor(cat_prod_edge_df[['product_node_id', 'cat_node_id']].values.tolist()).t().contiguous()

#graph["family", "fam_outfit", "family"].edge_index = torch.as_tensor(outfits_edges_df[['family_p1_node_id', 'family_p2_node_id']].values.tolist()).t().contiguous()

#graph["category", "cat_outfit", "category"].edge_index = torch.as_tensor(outfits_edges_df[['category_p1_node_id', 'category_p2_node_id']].values.tolist()).t().contiguous()

graph["product", "product_outfit", "product"].edge_index = torch.as_tensor(outfits_edges_df[['p1_node_id', 'p2_node_id']].values.tolist()).t().contiguous()


In [10]:
graph


HeteroData(
  category={
    num_nodes=129,
    node_id=[129],
  },
  product={
    num_nodes=398670,
    node_id=[398670],
    category_id=[398670],
    family_id=[398670],
  },
  (product, cat_prod_link, category)={ edge_index=[2, 398670] },
  (product, product_outfit, product)={ edge_index=[2, 2449219] }
)

In [11]:
graph = T.ToUndirected()(graph)

In [12]:
graph

HeteroData(
  category={
    num_nodes=129,
    node_id=[129],
  },
  product={
    num_nodes=398670,
    node_id=[398670],
    category_id=[398670],
    family_id=[398670],
  },
  (product, cat_prod_link, category)={ edge_index=[2, 398670] },
  (product, product_outfit, product)={ edge_index=[2, 3280676] },
  (category, rev_cat_prod_link, product)={ edge_index=[2, 398670] }
)

In [13]:
graph.validate()

True

In [14]:
graph.num_nodes, graph.num_edges

(398799, 4078016)

In [15]:
graph.node_types, graph.edge_types

(['category', 'product'],
 [('product', 'cat_prod_link', 'category'),
  ('product', 'product_outfit', 'product'),
  ('category', 'rev_cat_prod_link', 'product')])

In [16]:
graph.has_isolated_nodes()

False

In [17]:
torch.save(graph, 'data/graph/graph.pt')

## Embeddings

run [embeddings.py](embeddings.py) file to generate text emebddings using a pretrained model

In [ ]:
df = pd.read_parquet("text_embedding/")
df.head()

In [ ]:
df.dtypes

In [ ]:
df.to_parquet("data/product_text_embedding.parquet")

In [ ]:
torch.from_numpy(df.drop('product_node_id', axis=1).values)